In [1]:
import os
import sys

In [2]:
from dataclasses import dataclass # No need of constructor for a class
from pathlib import Path

@dataclass( frozen=True) # No changes can be made externally
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [ ]:
from ChestCancer.constants import * # CONFIG_FILE_PATH, PARAM_FILE_PATH
from ChestCancer.utils.common import read_yaml, create_directories
from ChestCancer import logger

In [11]:
class ConfigManager():

    """
    Manages project configuration by loading YAML files and providing
    component-specific configuration objects.
    """

    def __init__(self, config_filepath = CONFIG_FILE_PATH, param_filepath = PARAM_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.param = read_yaml(param_filepath)

        logger.info("Creating artifacts_root directory")
        create_directories([self.config.artifacts_root]) 


    def get_data_ingestion(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir= config.root_dir,
            source_URL= config.source_URL,
            local_data_file= config.local_data_file,
            unzip_dir= config.unzip_dir
        )
    
        return data_ingestion_config




In [5]:
from ChestCancer.utils.common import get_size
import zipfile
import gdown

In [17]:
class DataIngestion:

    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):

        try:
            data_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs(os.path.dirname(zip_download_dir), exist_ok=True)
            logger.info(f"Downloading data from {data_url} to {zip_download_dir}")

            file_id = self.config.source_URL.split("/")[-2]
            url = f"https://drive.google.com/uc?id={file_id}"

            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            output_zip = "artifacts/data_ingestion/data.zip"

            gdown.download(url, output_zip, quiet=False)

        except Exception as e:
            raise e

    
    def extract_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)

        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip:
            zip.extractall(unzip_path)





In [18]:
# Pipeline

try:
    config = ConfigManager()
    config_data_ingestion = config.get_data_ingestion()
    
    data_ingestion = DataIngestion(config_data_ingestion)
    data_ingestion.download_file()
    data_ingestion.extract_file()
except Exception as e:
    raise e


[2026-10-02 18:31:18,510: INFO: common: yaml file: [PosixPath('config/config.yaml')]]
[2026-10-02 18:31:18,513: INFO: common: yaml file: [PosixPath('params.yaml')]]
[2026-10-02 18:31:18,514: INFO: 3118583954: Creating artifacts_root directory]
[2026-10-02 18:31:18,516: INFO: common: created directory at: {path}]
[2026-10-02 18:31:18,517: INFO: common: created directory at: {path}]
[2026-10-02 18:31:18,519: INFO: 1832664527: Downloading data from https://drive.google.com/file/d/1H4hFsXxeo6i5IavjQ0MeLLRC9CZnf_Hw/view?usp=sharing to artifacts/data_ingestion/data.zip]


Downloading...
From: https://drive.google.com/uc?id=1H4hFsXxeo6i5IavjQ0MeLLRC9CZnf_Hw
To: /home/shlok/Personal_Projects/chest-cancer-classifier/Chest-Cancer-Classifier/artifacts/data_ingestion/data.zip
100%|██████████| 13.5M/13.5M [00:01<00:00, 7.67MB/s]
